In [0]:
-- ============================================================================
-- DATABRICKS TFL SQL SETUP: Raw Data Lake for dbt Transformation Practice
-- ============================================================================
-- Purpose: Create raw.tflsql database with 10K messy dummy records
-- Use Case: Practice dbt transforms, data quality checks, business queries
-- Architecture: Azure Databricks Workspace > Catalog > Schema > Tables
-- ============================================================================

-- STEP 1: Set workspace context and create catalog (if needed)
-- ============================================================================
-- Run this in a Databricks SQL cell

USE CATALOG main;  -- Default catalog in Databricks; use your prod catalog name if different

-- STEP 2: Create raw schema (database)
-- ============================================================================
-- This mimics a real data lake structure: main.raw schema holds all ingested data

CREATE SCHEMA IF NOT EXISTS raw
COMMENT 'Raw data lake - messy ingested API data before transformation'
WITH DBPROPERTIES (
    'owner' = 'data_engineering',
    'created_date' = '2025-05-05',
    'environment' = 'dev',
    'retention_days' = '90'
);

-- STEP 3: Create tflsql raw table with full schema
-- ============================================================================
-- Schema mirrors real TFL API response with common data quality issues:
-- - NULL values in station_name, line_name
-- - Negative passenger counts (data entry errors)
-- - Duplicate records from API reingestion
-- - Whitespace inconsistencies in station names
-- - Data type mismatches (strings that should be numeric)
-- - Outlier revenue values
-- - Future-dated timestamps (API test data leakage)

DROP TABLE IF EXISTS raw.tflsql;

CREATE TABLE raw.tflsql (
    event_id STRING COMMENT 'Unique tap event ID from TFL API',
    station_name STRING COMMENT 'Station name (HAS NULLS, whitespace issues)',
    line_name STRING COMMENT 'Tube line name (HAS NULLS, mixed case)',
    tap_timestamp TIMESTAMP COMMENT 'When passenger tapped card (HAS FUTURE DATES)',
    passenger_count INT COMMENT 'Number passengers in group (HAS NEGATIVES, outliers)',
    card_type STRING COMMENT 'Oyster, Contactless, Paper (HAS UNKNOWN values)',
    fare_zone INT COMMENT 'TFL zone 1-6 (HAS OUT-OF-RANGE values)',
    revenue DECIMAL(10, 2) COMMENT 'Transaction amount GBP (HAS DUPLICATES across events)',
    _file_name STRING COMMENT 'Source file name for dup detection',
    _load_date TIMESTAMP COMMENT 'When this record was ingested',
    _load_id STRING COMMENT 'Load batch ID for rerun dedup'
)
USING DELTA
COMMENT 'Raw TFL tap events from api.tfl.gov.uk - messy for practice'
PARTITIONED BY (_load_date)
TBLPROPERTIES (
    'delta.dataChangeFile' = 'true',
    'delta.deletedFileRetentionDuration' = '30 days',
    'owner' = 'data_engineering_team'
);

-- STEP 4: Create helper table for metadata tracking
-- ============================================================================
-- Track every load attempt for audit and dup detection

CREATE TABLE IF NOT EXISTS raw.tflsql_load_audit (
    load_id STRING,
    file_name STRING,
    load_date TIMESTAMP,
    record_count INT,
    min_timestamp TIMESTAMP,
    max_timestamp TIMESTAMP,
    null_count INT,
    duplicate_count INT,
    load_status STRING COMMENT 'SUCCESS, FAILED, PARTIAL'
)
USING DELTA;

-- STEP 5: Insert 10,000 dummy records with realistic data quality issues
-- ============================================================================
-- IMPORTANT: Run this in a NEW SQL cell to avoid timeout
-- This insert demonstrates:
-- 1. Realistic TFL data distribution
-- 2. Common data quality issues (nulls, duplicates, out-of-range)
-- 3. Schema drift (extra columns, missing data)
-- 4. Business logic inconsistencies (revenue not matching zone)

INSERT INTO raw.tflsql
SELECT
    -- Generate event_id: UUID-like string
    CONCAT('EV_', SUBSTRING(MD5(RAND()), 1, 12), '_', ROW_NUMBER() OVER (ORDER BY (SELECT NULL))) AS event_id,
    
    -- station_name: 70% valid, 20% NULL, 10% whitespace issues
    CASE 
        WHEN RAND() < 0.70 THEN
            CASE (FLOOR(RAND() * 25) % 25)
                WHEN 0 THEN 'King\'s Cross St Pancras'
                WHEN 1 THEN 'Waterloo'
                WHEN 2 THEN 'Liverpool Street'
                WHEN 3 THEN 'Victoria'
                WHEN 4 THEN 'London Bridge'
                WHEN 5 THEN 'Paddington'
                WHEN 6 THEN 'Euston'
                WHEN 7 THEN 'Oxford Circus'
                WHEN 8 THEN 'Piccadilly Circus'
                WHEN 9 THEN 'Leicester Square'
                WHEN 10 THEN 'Covent Garden'
                WHEN 11 THEN 'Bank'
                WHEN 12 THEN 'Monument'
                WHEN 13 THEN 'Tower Gateway'
                WHEN 14 THEN 'South Kensington'
                WHEN 15 THEN '  Baker Street  ' -- Whitespace issues
                WHEN 16 THEN 'bond street' -- Lowercase
                WHEN 17 THEN 'HAMMERSMITH' -- Uppercase
                WHEN 18 THEN 'Bethnal Green'
                WHEN 19 THEN 'Stratford'
                WHEN 20 THEN 'Canary Wharf'
                WHEN 21 THEN 'Jubilee'
                WHEN 22 THEN 'Camden Town'
                WHEN 23 THEN 'Angel'
                WHEN 24 THEN 'Moorgate'
                ELSE 'Unknown Station'
            END
        WHEN RAND() < 0.90 THEN NULL -- 20% null values
        ELSE CONCAT('  ', 'Station', FLOOR(RAND() * 100))  -- Malformed
    END AS station_name,
    
    -- line_name: 75% valid, 15% NULL, 10% invalid/mixed case
    CASE
        WHEN RAND() < 0.75 THEN
            CASE (FLOOR(RAND() * 12) % 12)
                WHEN 0 THEN 'Central'
                WHEN 1 THEN 'Circle'
                WHEN 2 THEN 'District'
                WHEN 3 THEN 'Northern'
                WHEN 4 THEN 'Bakerloo'
                WHEN 5 THEN 'Jubilee'
                WHEN 6 THEN 'Metropolitan'
                WHEN 7 THEN 'Victoria'
                WHEN 8 THEN 'Piccadilly'
                WHEN 9 THEN 'hammersmith & city' -- Lowercase
                WHEN 10 THEN 'DLRFLO' -- Abbreviation
                WHEN 11 THEN 'TfL Rail'
                ELSE 'Unknown Line'
            END
        WHEN RAND() < 0.90 THEN NULL -- 15% null
        ELSE CONCAT('LINE_', FLOOR(RAND() * 100)) -- Invalid format
    END AS line_name,
    
    -- tap_timestamp: Mostly recent (last 60 days), some future-dated (test data), some very old
    CASE
        WHEN RAND() < 0.85 THEN
            CURRENT_TIMESTAMP - INTERVAL CAST(FLOOR(RAND() * 60) AS INT) DAYS + 
            INTERVAL CAST(FLOOR(RAND() * 24) AS INT) HOURS
        WHEN RAND() < 0.95 THEN
            CURRENT_TIMESTAMP + INTERVAL CAST(FLOOR(RAND() * 30) AS INT) DAYS -- Future dates (test data leak)
        ELSE
            CURRENT_TIMESTAMP - INTERVAL CAST(FLOOR(RAND() * 365) AS INT) DAYS -- Old records
    END AS tap_timestamp,
    
    -- passenger_count: Range 1-300 for valid, but 20% have issues (negatives, outliers)
    CASE
        WHEN RAND() < 0.80 THEN CAST(FLOOR(RAND() * 300) + 1 AS INT) -- Valid 1-300
        WHEN RAND() < 0.90 THEN CAST(FLOOR(RAND() * 50) - 25 AS INT) -- Negative values (errors)
        WHEN RAND() < 0.95 THEN CAST(FLOOR(RAND() * 5000) AS INT) -- Large outliers
        ELSE 0 -- Zero passengers (edge case)
    END AS passenger_count,
    
    -- card_type: 85% valid, 15% unknown/invalid
    CASE
        WHEN RAND() < 0.50 THEN 'Contactless'
        WHEN RAND() < 0.75 THEN 'Oyster'
        WHEN RAND() < 0.85 THEN 'Paper'
        WHEN RAND() < 0.93 THEN 'Unknown'
        WHEN RAND() < 0.96 THEN 'Visa'  -- Invalid
        WHEN RAND() < 0.98 THEN 'MasterCard'  -- Invalid
        ELSE 'NULL'  -- String null
    END AS card_type,
    
    -- fare_zone: Range 1-6, but 10% out of range
    CASE
        WHEN RAND() < 0.90 THEN CAST(FLOOR(RAND() * 6) + 1 AS INT) -- Valid 1-6
        WHEN RAND() < 0.95 THEN CAST(FLOOR(RAND() * 10) + 7 AS INT) -- Out of range high
        ELSE CAST(FLOOR(RAND() * 5) - 3 AS INT) -- Out of range low/negative
    END AS fare_zone,
    
    -- revenue: Typically 1.50-5.00 GBP, but business rule inconsistencies
    -- (higher zone should = higher revenue, but often doesn't in raw data)
    CASE
        WHEN RAND() < 0.85 THEN ROUND(CAST(RAND() * 5.00 + 1.50 AS DECIMAL(10, 2)), 2)
        WHEN RAND() < 0.92 THEN ROUND(CAST(RAND() * 10000.00 AS DECIMAL(10, 2)), 2) -- Outliers
        WHEN RAND() < 0.98 THEN ROUND(CAST(RAND() * -5.00 AS DECIMAL(10, 2)), 2) -- Negative revenue
        ELSE 0.00 -- Zero revenue
    END AS revenue,
    
    -- _file_name: Track source; same file loaded multiple times = duplicates
    CASE
        WHEN RAND() < 0.85 THEN 'tfl_api_2025_05_05.csv'
        WHEN RAND() < 0.92 THEN 'tfl_api_2025_05_05.csv' -- Reingested, causing dupes
        WHEN RAND() < 0.96 THEN 'tfl_api_2025_05_04.csv'
        ELSE 'tfl_api_2025_05_03.csv'
    END AS _file_name,
    
    -- _load_date: When ingested
    CURRENT_TIMESTAMP AS _load_date,
    
    -- _load_id: Load batch identifier for dedup
    'LOAD_20250505_001' AS _load_id

FROM (
    -- Generate 10,000 rows
    SELECT ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS row_num
    FROM (
        SELECT 1 
        FROM (SELECT 1) t1
        CROSS JOIN (SELECT 1) t2
        CROSS JOIN (SELECT 1) t3
        CROSS JOIN (SELECT 1) t4
        CROSS JOIN (SELECT 1) t5
        CROSS JOIN (SELECT 1) t6
        CROSS JOIN (SELECT 1) t7
        CROSS JOIN (SELECT 1) t8
        CROSS JOIN (SELECT 1) t9
        CROSS JOIN (SELECT 1) t10
        CROSS JOIN (SELECT 1) t11
        CROSS JOIN (SELECT 1) t12
        CROSS JOIN (SELECT 1) t13
        CROSS JOIN (SELECT 1) t14
    ) AS unlimited_rows
    LIMIT 10000
);

-- STEP 6: Verify the load
-- ============================================================================

SELECT COUNT(*) AS total_records FROM raw.tflsql;

-- Check for quality issues
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN event_id IS NULL THEN 1 ELSE 0 END) AS null_event_ids,
    SUM(CASE WHEN station_name IS NULL THEN 1 ELSE 0 END) AS null_stations,
    SUM(CASE WHEN line_name IS NULL THEN 1 ELSE 0 END) AS null_lines,
    SUM(CASE WHEN passenger_count < 0 THEN 1 ELSE 0 END) AS negative_passengers,
    SUM(CASE WHEN passenger_count > 300 THEN 1 ELSE 0 END) AS outlier_passengers,
    SUM(CASE WHEN fare_zone < 1 OR fare_zone > 6 THEN 1 ELSE 0 END) AS invalid_zones,
    SUM(CASE WHEN revenue < 0 THEN 1 ELSE 0 END) AS negative_revenue,
    SUM(CASE WHEN revenue > 100 THEN 1 ELSE 0 END) AS outlier_revenue
FROM raw.tflsql;

-- STEP 7: Create a view for dbt staging (optional, but good practice)
-- ============================================================================
-- This view exposes the raw data; dbt will build on top of this

CREATE OR REPLACE VIEW raw.tflsql_stg AS
SELECT
    event_id,
    TRIM(UPPER(station_name)) AS station_name,
    TRIM(UPPER(line_name)) AS line_name,
    tap_timestamp,
    passenger_count,
    card_type,
    fare_zone,
    revenue,
    _file_name,
    _load_date,
    _load_id
FROM raw.tflsql;

-- ============================================================================
-- READY FOR dbt TRANSFORMATION
-- ============================================================================
-- Next Steps:
-- 1. Create dbt project in your Databricks workspace
-- 2. Build dbt models that reference raw.tflsql
-- 3. Implement data quality checks (great_expectations or dbt tests)
-- 4. Transform raw data into clean gold layer tables
-- 5. Answer business questions using dbt-built tables
--
-- Sample dbt model (models/stg/stg_tfl_taps.sql):
-- ============================================================================
-- {{ config(
--     materialized = 'view',
--     schema = 'refined'
-- ) }}
--
-- WITH source_data AS (
--     SELECT
--         event_id,
--         TRIM(UPPER(station_name)) as station_name,
--         TRIM(UPPER(line_name)) as line_name,
--         tap_timestamp,
--         passenger_count,
--         card_type,
--         fare_zone,
--         revenue,
--         _load_date
--     FROM {{ source('raw', 'tflsql') }}
--     WHERE passenger_count BETWEEN 1 AND 300  -- Clean outliers
--       AND fare_zone BETWEEN 1 AND 6           -- Valid zones
--       AND tap_timestamp <= CURRENT_TIMESTAMP  -- No future dates
-- ),
--
-- deduplicated AS (
--     SELECT
--         *,
--         ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY _load_date DESC) as rn
--     FROM source_data
-- )
--
-- SELECT
--     event_id,
--     station_name,
--     line_name,
--     tap_timestamp,
--     passenger_count,
--     card_type,
--     fare_zone,
--     revenue,
--     _load_date
-- FROM deduplicated
-- WHERE rn = 1;
--
-- ============================================================================
-- BUSINESS QUERY: Top 5 Stations by Passenger Volume (Last 60 Days)
-- ============================================================================

SELECT
    station_name,
    SUM(passenger_count) AS total_passengers,
    COUNT(*) AS tap_events,
    ROUND(AVG(passenger_count), 2) AS avg_passengers_per_event,
    ROW_NUMBER() OVER (ORDER BY SUM(passenger_count) DESC) AS rank
FROM raw.tflsql
WHERE
    station_name IS NOT NULL
    AND passenger_count BETWEEN 1 AND 300
    AND tap_timestamp >= CURRENT_TIMESTAMP - INTERVAL '60 days'
    AND tap_timestamp <= CURRENT_TIMESTAMP
GROUP BY station_name
HAVING COUNT(*) > 10  -- Minimum sample size
ORDER BY total_passengers DESC
LIMIT 5;

-- ============================================================================
-- dbt TEST SUITE (to include in your dbt_project.yml)
-- ============================================================================
-- Create a file: tests/generic/data_quality.sql
--
-- - name: raw_tflsql
--   description: Raw TFL tap events
--   columns:
--     - name: event_id
--       description: Unique event ID
--       tests:
--         - unique
--         - not_null
--     - name: station_name
--       description: Station name
--       tests:
--         - not_null
--     - name: passenger_count
--       description: Passenger count
--       tests:
--         - dbt_utils.expression_is_true:
--             expression: "passenger_count BETWEEN 1 AND 300"
--     - name: tap_timestamp
--       description: Tap timestamp
--       tests:
--         - dbt_utils.expression_is_true:
--             expression: "tap_timestamp <= CURRENT_TIMESTAMP"